# PySyft Owner Workflow — Final Spatial Health Analysis

This notebook initializes the local PySyft Datasite, uploads the protected health point data and the reference population grid, registers the researcher, and approves submitted code requests.

**Role separation:** the owner does not define analytical functions. The researcher submits analytical code as project-based code requests; the owner reviews and approves execution.

## 0. Start Datasite and log in as Owner

Run this notebook first. `reset=True` creates a clean, reproducible Datasite state. It deletes earlier datasets, users, projects, and requests.

In [ ]:
import syft as sy

print("PySyft version:", sy.__version__)

data_site = sy.orchestra.launch(
    name="health-datasite",
    reset=True,
    port=8093,
)

owner = data_site.login(
    email="info@openmined.org",
    password="changethis",
)

owner

## 1. Load protected health point dataset

The health CSV is semicolon-separated and uses decimal commas. The coordinates are stored as `lat`/`long` in geographic coordinates (`EPSG:4326`). Spatial analyses transform them to the grid CRS (`EPSG:23700`).

In [ ]:
import pandas as pd
import numpy as np

HEALTH_PATH = "data/2008_eov_illness_stand.csv"
POINT_INPUT_CRS = "EPSG:4326"
ANALYSIS_CRS = "EPSG:23700"

df_real = pd.read_csv(
    HEALTH_PATH,
    sep=";",
    decimal=",",
    encoding="utf-8-sig",
)

df_real.columns = [c.strip() for c in df_real.columns]

for col in ["lat", "long", "VISITS", "AGE", "Age"]:
    if col in df_real.columns:
        df_real[col] = pd.to_numeric(df_real[col], errors="coerce")

required_cols = {"lat", "long", "VISITS"}
missing = required_cols - set(df_real.columns)
if missing:
    raise ValueError(f"Missing required health columns: {missing}. Available columns: {list(df_real.columns)}")

if df_real[["lat", "long"]].dropna().empty:
    raise ValueError("lat/long columns contain no valid coordinates. Check CSV separator and decimal settings.")

print("Health data shape:", df_real.shape)
print("Health data columns:", df_real.columns.tolist())
print(df_real.dtypes)
display(df_real.head())
display(df_real[["lat", "long"]].describe())

## 2. Create mock health data

The mock dataset preserves the schema and valid coordinate range so the researcher can develop and test analysis code without direct access to the private records.

In [ ]:
rng = np.random.default_rng(42)

df_mock = df_real.copy()

# Remove direct identifiers from the mock copy if present.
# The private data remain unchanged in df_real.
for col in ["ID", "Dateofbirt", "Dateofbirth", "DateOfBirth"]:
    if col in df_mock.columns:
        df_mock = df_mock.drop(columns=col)

# Keep WGS84 coordinates valid but slightly perturbed.
df_mock["lat"] = df_real["lat"] + rng.normal(0, 0.001, len(df_real))
df_mock["long"] = df_real["long"] + rng.normal(0, 0.001, len(df_real))

# Perturb numeric non-coordinate variables for development/testing.
for col in df_mock.select_dtypes(include=[np.number]).columns:
    if col in ["lat", "long"]:
        continue
    df_mock[col] = df_mock[col] + rng.normal(0, 1.0, len(df_mock))

if "VISITS" in df_mock.columns:
    df_mock["VISITS"] = df_mock["VISITS"].clip(lower=0)

print("Mock health data shape:", df_mock.shape)
display(df_mock.head())
display(df_mock[["lat", "long"]].describe())

## 3. Upload health dataset as PySyft asset

In [ ]:
health_asset = sy.Asset(
    name="health_distribution_asset",
    data=df_real,
    mock=df_mock,
    description=(
        "Protected health point dataset. Coordinates are WGS84 longitude/latitude "
        "in columns 'long' and 'lat'."
    ),
)

health_dataset = sy.Dataset(
    name="Health Distribution Dataset",
    summary="Protected original health point dataset with mock counterpart.",
    description="Private data remain on the Datasite. Only approved outputs are returned.",
    citation="Internal dataset",
)

health_dataset.add_asset(health_asset)
owner.upload_dataset(health_dataset)

owner.datasets

## 4. Load and upload the population grid

PySyft cannot upload a `GeoDataFrame` directly. The grid is therefore converted to a regular `pandas.DataFrame` with polygon geometries stored as WKT strings in `geometry_wkt`. Syft functions reconstruct the GeoDataFrame internally.

In [ ]:
import geopandas as gpd

GRID_PATH = "data/GHS_Hungary_pop_city-clipped.shp"

grid_gdf = gpd.read_file(GRID_PATH)
if grid_gdf.crs is None:
    grid_gdf = grid_gdf.set_crs(ANALYSIS_CRS)
else:
    grid_gdf = grid_gdf.to_crs(ANALYSIS_CRS)

grid_crs = str(grid_gdf.crs)

grid_df = grid_gdf.copy()
grid_df["geometry_wkt"] = grid_df.geometry.to_wkt()
grid_df = pd.DataFrame(grid_df.drop(columns="geometry"))

for col in ["GID", "VALUE", "pop_dens"]:
    if col in grid_df.columns:
        grid_df[col] = pd.to_numeric(grid_df[col], errors="coerce")

required_grid_cols = {"GID", "VALUE", "geometry_wkt"}
missing_grid = required_grid_cols - set(grid_df.columns)
if missing_grid:
    raise ValueError(f"Missing required grid columns: {missing_grid}. Available columns: {list(grid_df.columns)}")

print("Grid CRS:", grid_crs)
print("Grid shape:", grid_df.shape)
print("Grid columns:", grid_df.columns.tolist())
display(grid_df.head())

In [ ]:
# The population grid is a reference layer. For reproducible spatial tests,
# the full grid is used as mock. If the grid is sensitive in another setting,
# replace this with a reduced or synthetic mock grid.
grid_mock = grid_df.copy()

grid_asset = sy.Asset(
    name="hungary_population_grid",
    data=grid_df,
    mock=grid_mock,
    description=(
        "Population grid clipped to the study area. "
        f"CRS: {ANALYSIS_CRS}. Geometry stored as WKT in column 'geometry_wkt'. "
        "Expected columns: GID, VALUE, pop_dens, geometry_wkt."
    ),
)

grid_dataset = sy.Dataset(
    name="Hungary Population Grid",
    summary="Reference population grid used for privacy-preserving spatial aggregation.",
    description="Grid geometry is stored as WKT for PySyft compatibility.",
    citation="Internal / external reference grid",
)

grid_dataset.add_asset(grid_asset)
owner.upload_dataset(grid_dataset)

print("Population grid uploaded successfully.")
owner.datasets

## 5. Register researcher

In [ ]:
owner.register(
    email="researcher@example.com",
    name="Researcher",
    password="pw",
    password_verify="pw",
)

## 6. Inspect submitted requests

Run this section after the researcher notebook has submitted code requests.

In [ ]:
owner.refresh()

for i, req in enumerate(owner.requests):
    print("INDEX:", i)
    print(req)
    print("-" * 80)

## 7. Review request code

In [ ]:
owner.refresh()

for i, req in enumerate(owner.requests):
    print(f"Request index {i}")
    try:
        print(req.code)
    except Exception as e:
        print("Could not display code:", type(e).__name__, e)
    print("-" * 80)

## 8. Approve requests

For the controlled thesis demonstration, this cell attempts to approve all currently pending requests. In a real data-governance setting, approve only after manual code review.

In [ ]:
owner.refresh()

for i, req in enumerate(owner.requests):
    try:
        result = req.approve()
        print(f"Approved request {i}:", result)
    except Exception as e:
        print(f"Skipped request {i}:", type(e).__name__, e)

owner.refresh()
owner.requests